In [0]:
from typing import Any

In [0]:
import logging

from py4j.java_gateway import JavaObject  # type: ignore
from pyspark.sql import SparkSession
from pyspark.sql.window import Window
from assignment_internal import AssignmentInternal
# from assignment_internal import AssignmentInternal
from pyspark.sql.functions import (
    coalesce,
    concat_ws,
    current_timestamp,
    from_utc_timestamp,
    lit,
    lower,
    sha2,
    when,
)

In [0]:
dbutils.widgets.text('curated_zone', 'None', '01-Curated_zone')
dbutils.widgets.text('catalog_landed', 'None', '02-Catalog_landed')
dbutils.widgets.text('reference_zone', 'None', '03-Reference_zone')
dbutils.widgets.text('run_date', 'None', '04-Run Date')
dbutils.widgets.text('schema', 'None', '05-Schema')
dbutils.widgets.text('env_prefix', 'None', '06-Environment')
curated_zone = dbutils.widgets.get('curated_zone')
catalog_landed = dbutils.widgets.get('catalog_landed')
reference_zone = dbutils.widgets.get('reference_zone')
run_date = dbutils.widgets.get('run_date')
schema = dbutils.widgets.get('schema')
env_prefix = dbutils.widgets.get('env_prefix')

import sys, os
relative_path = '../../helpers/lng_lnd'
path = os.path.abspath(relative_path)
if path not in sys.path:
  sys.path.append(path)
from landing_helpers import *
from landed_helpers import *
relative_path = '../../helpers/curated_helpers'
path = os.path.abspath(relative_path)
if path not in sys.path:
  sys.path.append(path)
from delta_table_creator import *
from extract import *
from transform import *
spark = SparkSession.builder.getOrCreate()


In [0]:
def load_curated_assignment_qc(
    spark, curated_zone, catalog_landed, reference_zone, run_date, schema, env_prefix
) -> None:
    args = [curated_zone, catalog_landed, reference_zone, run_date, schema, env_prefix]
    if all(item is not None for item in args) :
        date_filter = f"businessdate = {run_date!r}"

        # schema = (
        #      "AssignmentId String,"
        #      "SourceSystem String,"
        #      "PortfolioKey String,"
        #      "PortfolioName String,"
        #      "IngestionTimestamp timestamp,"
        #      "BatchId String,"
        #      "AssignmentKey String,"
        #      "CaseCode Int,"
        #      "ExposureKey String,"
        #      "PartyKey String, "
        #      "CDI String,"
        #      "TaxIdentificationNumber String,"
        #      "AgreementKey String,"
        #      "InternalAssignmentStartDate String,"
        #      "ExternalAssignmentStartDate String,"
        #      "AssignmentRecallDate timestamp,"
        #      "ChannelCode String,"
        #      "Channel String,"
        #      "ExternalCompanyAgency String,"
        #      "ExternalCompanyPartyKey String,"
        #      "ExternalCompanyFullName String,"
        #      "AssignmentTypeCode String,"
        #      "AssignmentType String,"
        #      "TeamCode String,"
        #      "Team String,"
        #      "AgentPartyKey String,"
        #      "AgentFullName String,"
        #      "Supervisor String,"
        #      "AssistantManager String,"
        #      "Manager String,"
        #      "Head String,"
        #      "AgentFirstName String,"
        #      "AssignmentBucketCode Int,"
        #      "AssignmentBucket String,"
        #      "AssignmentProcess String,"
        #      "ProcessType String,"
        #      "PacketCode Int,"
        #      "Packet String,"
        #      "PacketPreviousEndDate timestamp,"
        #      "PacketAssignmentDate timestamp,"
        #      "PaccCode Int,"
        #      "QueCode Int,"
        #      "Que String,"
        #      "Segment String,"
        #      "ExtraSegment String,"
        #      "StreamTypeCode Int,"
        #      "StreamType String,"
        #      "StreamStatusCode Int,"
        #      "StreamStatus String,"
        #      "StreamAssignedStatusCode Int,"
        #      "StreamAssignedStatus String,"
        #      "OutstandingAmount double,"
        #      "DebtAmount double,"
        #      "BusinessDate Date,"
        #     )


        cluster_columns = ["BusinessDate"]

        curated_table_path = f"{curated_zone}.Assignment"

        # ================== Create Delta Table ==================
        logging.info("Creating Delta Table...")
        if spark.catalog.tableExists(curated_table_path) is False:
            create_delta_table(
                spark,
                curated_table_path,
                cluster_columns,
                schema,  # type: ignore
            )
        else:
            logging.info("Table already exists.")

        # ================== Delete From Delta Table ==================
        if spark.catalog.tableExists(curated_table_path) is True:
            delete_from_curated_table(
                    spark,
                    curated_table_path,
                    run_date,  # type: ignore
                    "qc",
                )        
        else:
            logging.info("Tables doe not exist, no need to delete entries for business date!")

        # ================== portfolio ==================

        portfolio_extra_filters = None

        if portfolio_extra_filters is not None:
            portfolio_extra_filters.append(date_filter)
            portfolio_final_filters = " AND ".join(portfolio_extra_filters)
        else:
            portfolio_final_filters = date_filter  # possible need change date

        portfolio_selected_fields = [
            "PortfolioName",
            "Prefix",
            "vendor_code_qualco",
            "businessdate",
        ]

        portfolio = load_landed_portfolio_table(  # noqa: F841
            spark,
            curated_zone,  # type: ignore
            "portfolio",
            portfolio_final_filters,
            portfolio_selected_fields,
        )
        # ================== case_master ==================

        case_master_extra_filters = None

        if case_master_extra_filters is not None:
            case_master_extra_filters.append(date_filter)
            case_master_final_filters = " AND ".join(case_master_extra_filters)
        else:
            case_master_final_filters = date_filter

        case_master_selected_fields = [
            "case_code",
            "vend_code",
            "account_code",
            "old_case_code",
            "cust_code",
            "pack_code",
            "pack_name",
            "CACO_EMP_ASSIGNMENT_DATE",
            "pacc_dateback",
            "que_code",
            "que_desc",
            "businessdate",
        ]

        case_master = load_landed_delta_table(
            spark,
            catalog_landed,  # type: ignore
            "qc_case_master",  # check name
            case_master_final_filters,
            case_master_selected_fields,
        )

        # ================== customer ==================

        customer_extra_filters = None

        if customer_extra_filters is not None:
            customer_extra_filters.append(date_filter)
            customer_extra_filters = " AND ".join(customer_extra_filters)
        else:
            customer_extra_filters = date_filter

        customer_selected_fields = [
            "cust_code",
            "cdi",
            "tax_number",
            "businessdate",
        ]

        customer = load_landed_delta_table(
            spark,
            catalog_landed,  # type: ignore
            "qc_customer",  # check name
            customer_extra_filters,
            customer_selected_fields,
        )

        # ================== case_assignment ==================

        case_assignment_extra_filters = None

        if case_assignment_extra_filters is not None:
            case_assignment_extra_filters.append(date_filter)
            case_assignment_final_filters = " AND ".join(case_assignment_extra_filters)
        else:
            case_assignment_final_filters = date_filter

        case_assignment_selected_fields = [
            "agent_id",
            "assignment_process",
            "case_code",
            "channel_id",
            "customer_channel",
            "extc_code",
            "Extra_Segment",
            "final_bucket",
            "legal_channel_id",
            "process_type",
            "que_desc",
            "segment",
            "team_id",
            "businessdate",
        ]

        case_assignment = load_landed_delta_table(
            spark,
            catalog_landed,  # type: ignore
            "qc_case_assignment",  # check name
            case_assignment_final_filters,
            case_assignment_selected_fields,
        )

        # ================== employees ==================

        employees_extra_filters = None

        if employees_extra_filters is not None:
            employees_extra_filters.append(date_filter)
            employees_final_filters = " AND ".join(employees_extra_filters)
        else:
            employees_final_filters = date_filter

        employees_selected_fields = [
            "emp_Code",
            "emp_first_name",
            "emp_last_name",
            "emp_login",
            "emp_user_type",
            "emp_job_desc",
            "dep_code",
            "department",
            "team_id",
            "team_name",
            "supervisor_emp_code",
            "supervisor_login",
            "manager_emp_code",
            "manager_login",
            "emp_active",
            "created",
            "updated",
            "businessdate",
        ]

        employees = load_landed_delta_table(
            spark,
            catalog_landed,  # type: ignore
            "qc_employees",
            employees_final_filters,
            employees_selected_fields,
        )

        # ================== team ==================

        team_extra_filters = None

        if team_extra_filters is not None:
            team_extra_filters.append(date_filter)
            team_final_filters = " AND ".join(team_extra_filters)
        else:
            team_final_filters = date_filter

        team_selected_fields = [
            "TEA_CODE",
            "TEA_NAME",
            "Supervisor_emp_code",
            "Supervisor_login",
            "Channel_id",
            "Channel_name",
            "Manager_emp_code",
            "Manager_login",
            "created",
            "BusinessDate",
        ]

        team = load_landed_delta_table(
            spark,
            catalog_landed,  # type: ignore
            "qc_team",
            team_final_filters,
            team_selected_fields,
        )

        # ================== department ==================

        department_extra_filters = None

        if department_extra_filters is not None:
            department_extra_filters.append(date_filter)
            department_final_filters = " AND ".join(department_extra_filters)
        else:
            department_final_filters = date_filter

        department_selected_fields = [
            "dep_code",
            "dep_desc",
            "head_name",
            "head_emp_code",
            "BusinessDate",
        ]

        department = load_landed_delta_table(
            spark,
            catalog_landed,  # type: ignore
            "qc_department",
            department_final_filters,
            department_selected_fields,
        )

        # ================== external_companies ======================
        external_companies_extra_filters = None

        if external_companies_extra_filters is not None:
            external_companies_extra_filters.append(date_filter)
            external_companies_final_filters = " AND ".join(
                external_companies_extra_filters
            )
        else:
            external_companies_final_filters = date_filter

        external_companies_selected_fields = [
            "extc_code",
            "extc_name",
            "extc_intcode",
            "EXTC_COMMENT",
            "BusinessDate",
        ]

        external_companies = load_landed_delta_table(
            spark,
            catalog_landed,  # type: ignore
            "qc_external_companies",
            external_companies_final_filters,
            external_companies_selected_fields,
        )

        # ================== at_customer_core ==================

        at_customer_core_extra_filters = None

        if at_customer_core_extra_filters is not None:
            at_customer_core_extra_filters.append(date_filter)
            at_customer_core_final_filters = " AND ".join(at_customer_core_extra_filters)
        else:
            at_customer_core_final_filters = date_filter

        at_customer_core_selected_fields = [
            "CUCO_DEDICATED_PROPERTY_05_INTCODE",
            "cust_code",
            "businessdate",
        ]

        at_customer_core = load_landed_delta_table(
            spark,
            catalog_landed,  # type: ignore
            "qc_at_customer_core",
            at_customer_core_final_filters,
            at_customer_core_selected_fields,
        )

        # ================== lst_of_val ==================

        lst_of_val_extra_filters = None

        if lst_of_val_extra_filters is not None:
            lst_of_val_extra_filters.append(date_filter)
            lst_of_val_final_filters = " AND ".join(lst_of_val_extra_filters)
        else:
            lst_of_val_final_filters = date_filter

        lst_of_val_selected_fields = [
            "lov_code",
            "lov_desc",
            "businessdate",
        ]

        lst_of_val = load_landed_delta_table(
            spark,
            catalog_landed,  # type: ignore
            "qc_lst_of_val",
            lst_of_val_final_filters,
            lst_of_val_selected_fields,
        )

        # ================== external_companies__external_only ======================
        external_companies_external_only_extra_filters = None

        if external_companies_external_only_extra_filters is not None:
            external_companies_external_only_extra_filters.append(date_filter)
            external_companies_external_only_final_filters = " AND ".join(
                external_companies_external_only_extra_filters
            )
        else:
            external_companies_external_only_final_filters = date_filter

        external_companies_external_only_selected_fields = [
            "extc_code",
            "extc_name",
            "extc_intcode",
            "EXTC_COMMENT",
            "BusinessDate",
        ]

        external_companies_external_only = load_landed_delta_table(
            spark,
            catalog_landed,  # type: ignore
            "qc_external_companies",
            external_companies_external_only_final_filters,
            external_companies_external_only_selected_fields,
        )

        external_companies_external_only = (
            external_companies_external_only.withColumn(
                "extc_code_external", external_companies_external_only.extc_code
            )
            .withColumn(
                "extc_name_external", external_companies_external_only.extc_name
            )
            .withColumn(
                "EXTC_COMMENT_external", external_companies_external_only.EXTC_COMMENT
            )
        )

        # Keep only specific columns
        external_companies_external_only = external_companies_external_only.select(
            "extc_code_external", "extc_name_external", "EXTC_COMMENT_external"
        )


        # ================== case_external_assignment ================== 

        case_external_assignment_extra_filters = None

        if case_external_assignment_extra_filters is not None:
            case_external_assignment_extra_filters.append(date_filter)
            case_external_assignment_final_filters = " AND ".join(case_external_assignment_extra_filters)
        else:
            case_external_assignment_final_filters = date_filter

        case_external_assignment_selected_fields = [
            "case_code",
            "vend_code",
            "pack_assign_date",
            "PACC_OUTSTANDING_AMT",
            "PACC_DEBT_AMT",
            "pacc_code",
            ]

        case_external_assignment = load_landed_delta_table(
            spark,
            catalog_landed,  # type: ignore
            "qc_case_external_assignment",  # check name
            case_external_assignment_final_filters,
            case_external_assignment_selected_fields,)
        
        # ================== Subjoin =========================

        window_spec = Window.partitionBy(
            "case_code",
            "vend_code"
        ).orderBy(
            F.col("pack_assign_date").desc()
        )

        # Add a row number column within each partition to get the latest row per case_code and vend_code
        case_external_assignment_rn = (
            case_external_assignment.select(
                    "case_code",
                    "vend_code",
                    "pack_assign_date",
                    "pacc_outstanding_amt",
                    "pacc_debt_amt",
                    "pacc_code",
                )
                .withColumn("rn", F.row_number().over(window_spec))
                .where(F.col("rn") == 1)
                .drop("rn")
                .withColumnRenamed("case_code", "cea_case_code")
                .withColumnRenamed("vend_code", "cea_vend_code")
                .withColumnRenamed("pack_assign_date", "cea_pack_assign_date")
                .withColumnRenamed("pacc_outstanding_amt", "cea_pacc_outstanding_amt")
                .withColumnRenamed("pacc_debt_amt", "cea_pacc_debt_amt")
                .withColumnRenamed("pacc_code", "cea_pacc_code")
            )

        # ================== Join Conditions==================
        case_master_case_assignment_join_condition = (
            case_master["case_code"] == case_assignment["case_code"]
        )

        case_master_final_portfolio_join_condition = (
            case_master["vend_code"] == portfolio["vendor_code_qualco"]
        )

        case_master_final_customer_join_condition = (
            case_master["cust_code"] == customer["cust_code"]
        )

        case_master_final_employees_join_condition = (
            case_assignment["agent_id"] == employees["emp_Code"]
        )

        case_master_final_ext_companies_join_condition = (
            case_assignment["legal_channel_id"] == external_companies["extc_code"]
        )

        case_master_final_ext_companies_external_only_join_condition = (
            case_assignment["channel_id"]
            == external_companies_external_only["extc_code_external"]
        )

        case_assignment_team_join_condition = (
            case_assignment["team_id"] == team["TEA_CODE"]
        )

        case_assignment_department_join_condition = (
            case_assignment["channel_id"] == department["dep_code"]
        )

        case_master_at_customer_core_join_condition = (
            case_master["cust_code"] == at_customer_core["cust_code"]
        )

        lst_of_val_at_customer_core_join_condition = (
            lst_of_val["LOV_CODE"] == at_customer_core["CUCO_DEDICATED_PROPERTY_05_INTCODE"]
        )

        case_master_case_external_assignment_rn_join_condition = (
            (case_master["case_code"] == case_external_assignment_rn["cea_case_code"]) &
            (case_master["vend_code"] == case_external_assignment_rn["cea_vend_code"])
        )


        # Joins
        target_table_qc = (
            case_master.join(
                case_assignment.dropDuplicates(),
                case_master_case_assignment_join_condition,
                "inner",
            )
            .join(
                portfolio,
                case_master_final_portfolio_join_condition,
                "left",
            )
            .join(
                customer,
                case_master_final_customer_join_condition,
                "left",
            )
            .join(
                employees,
                case_master_final_employees_join_condition,
                "left",
            )
            .join(
                external_companies,
                case_master_final_ext_companies_join_condition,
                "left",
            )
            .join(
                external_companies_external_only,
                case_master_final_ext_companies_external_only_join_condition,
                "left",
            )
            .join(
                case_external_assignment_rn,
                case_master_case_external_assignment_rn_join_condition,
                "left",
            )            
            .join(
                team,
                case_assignment_team_join_condition,
                "left",
            )
            .join(
                department,
                case_assignment_department_join_condition,
                "left",
            )
            .join(
                at_customer_core,
                case_master_at_customer_core_join_condition,
                "left",
            )
            .join(
                lst_of_val,
                lst_of_val_at_customer_core_join_condition,
                "left",
            )
            
        )


        # HashKeys
        target_table_qc_AssignmentId_primary_key_fields = [
            case_master.case_code,
            lit("qc"),
        ]

        # target_table_qc_PartyId_primary_key_fields = [
        #    case_master.cust_code,
        #    lit("qc"),
        # ]

        # target_table_qc_ExposureId_primary_key_fields = [
        #    case_master.old_case_code,
        #    lit("qc"),
        # ]

        # target_table_qc_ExternalCompanyPartyId_primary_key_fields = [
        #    case_assignment.extc_code,
        #    lit("qc"),
        # ]

        # target_table_qc_AgentId_primary_key_fields = [
        #    case_assignment.agent_id,
        #    lit("qc"),
        # ]

        target_table_qc = target_table_qc.withColumn(
            "AssignmentId",
            sha2(
                concat_ws(";", *target_table_qc_AssignmentId_primary_key_fields),
                256,
            ),
        )
        #    .withColumn(
        #        "PartyId",
        #        sha2(
        #            concat_ws("", *target_table_qc_PartyId_primary_key_fields),
        #            256,
        #        ),
        #    )
        #    .withColumn(
        #        "ExposureId",
        #        sha2(
        #            concat_ws("", *target_table_qc_ExposureId_primary_key_fields),
        #            256,
        #        ),
        #    )
        #    .withColumn(
        #        "ExternalCompanyPartyId",
        #        sha2(
        #            concat_ws(
        #                "", *target_table_qc_ExternalCompanyPartyId_primary_key_fields
        #            ),
        #            256,
        #        ),
        #    )
        #    .withColumn(
        #        "AgentPartyId",
        #        sha2(
        #            concat_ws("", *target_table_qc_AgentId_primary_key_fields),
        #            256,
        #        ),
        #    )
        # )

        target_table_qc = (
            target_table_qc.withColumn(
                "LegalChannelCode",
                when(case_assignment.legal_channel_id == -1, lit(None)).otherwise(
                    coalesce(case_assignment.legal_channel_id, lit(None))
                ),
            )
            .withColumn(
                "AssignmentTeamCode",
                coalesce(case_assignment.team_id, lit(-1)),
            )
            .withColumn(
                "AgentPartyKey",
                when(
                    case_assignment.agent_id == -1, None
                ).otherwise(  # If assignmentteam is -1, set to null
                    case_assignment.agent_id
                ),  # Else return assignmentteam value
            )
            .withColumn(
                "EmployeeFullName",
                concat_ws(
                    " ",
                    coalesce(employees["emp_first_name"], lit("")),
                    coalesce(employees["emp_last_name"], lit("")),
                ),
            )
            .withColumn(
                "ChannelCode",
                when(
                    lower(case_assignment.assignment_process).isin(
                        "lf_offer",
                        "dnc_offer",
                        "collection",
                        "collection_judicial_escalation",
                    ),
                    lit("105"),
                ).otherwise(department["dep_code"]),
            )
            .withColumn(
                "Channel",
                when(
                    lower(case_assignment.assignment_process).isin(
                        "lf_offer",
                        "dnc_offer",
                        "collection",
                        "collection_judicial_escalation",
                    ),
                    lit("External LF"),
                ).otherwise(department["dep_desc"]),
            )
        )

        target_table_qc = target_table_qc.withColumn(
            "AssignmentType",
            when(
                lower(case_assignment.assignment_process).isin(
                    "lf_offer",
                    "dnc_offer",
                    "collection",
                    "collection_judicial_escalation",
                ),
                "External",
            )
            .when(
                (target_table_qc.AgentPartyKey.isNull())
                & (case_assignment["legal_channel_id"] == -1)
                & (case_assignment["channel_id"] == -1)
                & (case_assignment["team_id"] == -1),
                "Unassigned",
            )
            .when(
                (
                    (target_table_qc.AgentPartyKey.isNull())
                    | (case_assignment["team_id"] != -1)
                    | (case_assignment["channel_id"] != -1)
                )
                & (case_assignment["legal_channel_id"] == -1),
                "Internal",
            )
            .otherwise("Internal and External"),
        )

        target_table_qc = (
            target_table_qc.withColumn(
                "ExternalCompanyFullName",
                when(
                    target_table_qc.AssignmentType == "External",
                    target_table_qc.extc_name_external,
                ).otherwise(target_table_qc.extc_name),
            )
            .withColumn(
                "ExternalCompanyPartyKey",
                when(
                    target_table_qc.AssignmentType == "External",
                    target_table_qc.extc_code_external,
                ).otherwise(target_table_qc.LegalChannelCode),
            )
            .withColumn(
                "ExternalCompanyAgency",
                when(
                    target_table_qc.AssignmentType == "External",
                    target_table_qc.EXTC_COMMENT_external,
                ).otherwise(target_table_qc.EXTC_COMMENT),
            )
        )

        # ================== Selection of Final Columns ==================

        result_df_qc = target_table_qc.select(
            target_table_qc.AssignmentId.cast("String").alias("AssignmentId"),
            lit("qc").cast("String").alias("SourceSystem"),
            case_master.vend_code.cast("String").alias("PortfolioKey"),
            portfolio.PortfolioName.cast("String").alias("PortfolioName"),
            lit(from_utc_timestamp(current_timestamp(), "Europe/Athens")).alias(
                "IngestionTimestamp"
            ),
            lit(None).cast("String").alias("BatchId"),
            case_assignment.case_code.cast("String").alias("AssignmentKey"),
            case_assignment.case_code.cast("Int").alias("CaseCode"),
            case_master.old_case_code.cast("String").alias("ExposureKey"),
            case_master.cust_code.cast("String").alias("PartyKey"),
            customer.cdi.cast("String").alias("CDI"),
            customer.tax_number.cast("String").alias("TaxIdentificationNumber"),
            lit(None).cast("String").alias("AgreementKey"),
            case_master.CACO_EMP_ASSIGNMENT_DATE.cast("String").alias(
                "InternalAssignmentStartDate"
            ),
            lit(None).cast("String").alias("ExternalAssignmentStartDate"),
            lit(None).cast("timestamp").alias("AssignmentRecallDate"),
            target_table_qc.ChannelCode.cast("String").alias("ChannelCode"),
            target_table_qc.Channel.cast("String").alias("Channel"),
            target_table_qc.ExternalCompanyAgency.cast("String").alias(
                "ExternalCompanyAgency"
            ),
            target_table_qc.ExternalCompanyPartyKey.cast("String").alias(
                "ExternalCompanyPartyKey"
            ),
            target_table_qc.ExternalCompanyFullName.cast("string").alias(
                "ExternalCompanyFullName"
            ),
            target_table_qc.AssignmentType.cast("String").alias("AssignmentTypeCode"),
            target_table_qc.AssignmentType.cast("String").alias("AssignmentType"),
            team.TEA_CODE.cast("String").alias("TeamCode"),
            team.TEA_NAME.cast("String").alias("Team"),
            # case_assignment.agent_id.cast("String").alias("AssignmentAgentPartyKey"), #rename from AssignmentAgentPartyKey to AgentPartyKey
            target_table_qc.AgentPartyKey.cast("String").alias("AgentPartyKey"),
            target_table_qc.EmployeeFullName.cast("String").alias(
                "AgentFullName"
            ),  # change
            employees.emp_login.cast("String").alias("AgentLoginName"),
            lit(None).cast("String").alias("Supervisor"),
            lit(None).cast("String").alias("AssistantManager"),
            lit(None).cast("String").alias("Manager"),
            lit(None).cast("String").alias("Head"),
            case_assignment.final_bucket.cast("Int").alias("AssignmentBucketCode"),
            case_assignment.final_bucket.cast("String").alias("AssignmentBucket"),
            case_assignment.assignment_process.cast("String").alias(
                "AssignmentProcess"
            ),
            case_assignment.process_type.cast("String").alias("ProcessType"),
            case_master.pack_code.cast("Int").alias("PacketCode"),
            case_master.pack_name.cast("String").alias("Packet"),
            case_master.pacc_dateback.cast("timestamp").alias("PacketPreviousEndDate"),
            target_table_qc.cea_pack_assign_date.cast("timestamp").alias("PacketAssignmentDate"),
            target_table_qc.cea_pacc_code.cast("Int").alias("PaccCode"),
            case_master.que_code.cast("Int").alias("QueCode"),
            case_master.que_desc.cast("String").alias("Que"),
            case_assignment.segment.cast("String").alias("Segment"),
            case_assignment.Extra_Segment.cast("String").alias("ExtraSegment"),
            lit(None).cast("Int").alias("StreamTypeCode"),
            lit(None).cast("String").alias("StreamType"),
            lit(None).cast("Int").alias("StreamStatusCode"),
            lit(None).cast("String").alias("StreamStatus"),
            lit(None).cast("Int").alias("StreamAssignedStatusCode"),
            lit(None).cast("String").alias("StreamAssignedStatus"),
            target_table_qc.cea_pacc_outstanding_amt.cast("double").alias("OutstandingAmount"),
            target_table_qc.cea_pacc_debt_amt.cast("double").alias("DebtAmount"),
            lst_of_val.lov_desc.cast("String").alias("CampaignName"),
            case_master.businessdate.alias("BusinessDate"),
            lit(None).cast("Boolean").alias("ForcedOnboarding"),
            lit(None).cast("Boolean").alias("HolisticFlag"),
            lit(None).cast("String").alias("ForcedOnboardReason"),   
        )

        result_df_qc.write.format("delta").clusterBy("BusinessDate").mode(
            "append"
        ).saveAsTable(curated_table_path)

    else:
        null_args = [
            arg_name
            for arg_name, arg in args
            if arg
            in [
                curated_zone,
                catalog_landed,
                reference_zone,
                run_date,
                schema,
                env_prefix,
            ]
            and arg is None
        ]

        for arg_name in null_args:
            logging.error(f"No {arg_name} specified!")

    logging.info("We did it!")


In [0]:
load_curated_assignment_qc(spark, curated_zone, catalog_landed, reference_zone, run_date, schema, env_prefix)